# Edge Arabic TTS — Colab Training

Train **FastSpeech2-small** (~5.7M params) on ~1 hour of single-speaker Saudi MSA from [`HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel`](https://huggingface.co/datasets/HeshamHaroon/arabic-msa-25k-saudi-male-tashkeel).

**Before running:**
1. **Runtime → Change runtime type → Hardware accelerator: GPU (T4)**
2. Clone the repo or upload project files

Run `python data/download_hesham.py` then `python data/preprocess.py` (see COLAB.md).

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Verify GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    raise RuntimeError('No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 3 — Unzip project from Google Drive
# On your VM you ran:  cd ~ && zip -r sarj-colab.zip sarj -x "sarj/.venv/*" ...
import os, zipfile

ZIP_PATH = '/content/drive/MyDrive/edge-tts/sarj-colab.zip'
assert os.path.exists(ZIP_PATH), f'Upload zip to: {ZIP_PATH}'

!unzip -q -o "{ZIP_PATH}" -d /content/
%cd /content/sarj
!ls -la

In [ ]:
# OPTIONAL — Skip zip upload: download + preprocess on Colab (~15 min)
# Uncomment and run instead of Cell 3 if you did NOT upload data.

# !git clone <YOUR_GITHUB_REPO_URL> /content/sarj  # or upload code-only zip
# %cd /content/sarj
# !pip install -q -r requirements.txt
# !python data/download.py
# !python data/preprocess.py

In [ ]:
# Cell 4 — Install dependencies
%cd /content/sarj
!pip install -q -r requirements.txt

import pandas as pd
train = pd.read_csv('data/train.csv')
print(f'Train clips: {len(train)}, total hours: {train.duration_sec.sum()/3600:.2f}')

In [ ]:
# Cell 5 — Verify model + data pipeline
import yaml
from pathlib import Path
from data.text_normalize import load_vocab
from model.fastspeech import build_model
from data.dataset import TTSDataset, collate_batch

cfg = yaml.safe_load(open('configs/colab.yaml'))
vocab = load_vocab(Path('data/vocab.json'))
model = build_model(cfg, len(vocab))

ds = TTSDataset(Path('data/train.csv'), Path('data/vocab.json'))
batch = collate_batch([ds[0], ds[1]])
out = model(batch['tokens'], batch['token_lengths'], batch['mels'], batch['durations'])
print('mel_pred shape:', tuple(out['mel_pred'].shape))
print('Ready to train.')

In [ ]:
# Cell 6 — TRAIN (main step — ~2-4 hours on T4)
# Uses batch_size=2 in configs/colab.yaml (long ~50s clips need small batches)

import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/edge-tts/checkpoints', exist_ok=True)

!python train.py --config configs/colab.yaml --device cuda

In [ ]:
# Cell 7 — Copy checkpoints to Google Drive (survives disconnect)
import shutil, glob

DRIVE_CKPT = '/content/drive/MyDrive/edge-tts/checkpoints'
for f in glob.glob('checkpoints/*.pt'):
    shutil.copy2(f, DRIVE_CKPT)
    print('Copied', f)
print('Done. Download best.pt from Drive to your VM.')

In [ ]:
# Cell 8 — Plot training loss (optional)
import json
import matplotlib.pyplot as plt

steps, losses = [], []
with open('logs/train_log.jsonl') as f:
    for line in f:
        r = json.loads(line)
        steps.append(r['step'])
        losses.append(r['train_loss'])

plt.figure(figsize=(10, 4))
plt.plot(steps, losses)
plt.xlabel('Step')
plt.ylabel('Train loss')
plt.title('Training loss')
plt.grid(True)
plt.show()

In [ ]:
# Cell 9 — GPU RTF benchmark
!python eval/rtf_benchmark.py --device cuda --checkpoint checkpoints/best.pt

In [ ]:
# Cell 10 — Test synthesis on Colab
!python synthesize.py \
  --text "السَّلامُ عَلَيْكُمْ وَرَحْمَةُ اللهِ" \
  --out samples/colab_test.wav \
  --checkpoint checkpoints/best.pt \
  --device cuda

from IPython.display import Audio
Audio('samples/colab_test.wav')

## Resume training if Colab disconnected

```python
!python train.py --config configs/colab.yaml --device cuda --resume checkpoints/latest.pt
```

## Ablation (no PostNet) — optional, ~1-2h extra

```python
!python train.py --config configs/ablation_no_postnet.yaml --device cuda
```

## Back on your VM

1. Download `best.pt` from Google Drive → `~/sarj/checkpoints/best.pt`
2. `bash scripts/generate_samples.sh`
3. `python eval/rtf_benchmark.py --device cpu`